In [1]:
import numpy as np
import xarray as xr
import glob
import re
import matplotlib.pyplot as plt
from functools import partial


In [2]:
from dask.distributed import Client
client = Client(threads_per_worker=1)
client

Connection method: Cluster object,Cluster type: distributed.LocalCluster
Dashboard: https://jupyterhub.hpc.ucar.edu/stable/user/jjeffree/more_testing/proxy/8787/status,
Dashboard: https://jupyterhub.hpc.ucar.edu/stable/user/jjeffree/more_testing/proxy/8787/status,Workers: 16
Total threads: 16,Total memory: 120.00 GiB
Status: running,Using processes: True
Comm: tcp://127.0.0.1:38521,Workers: 16
Dashboard: https://jupyterhub.hpc.ucar.edu/stable/user/jjeffree/more_testing/proxy/8787/status,Total threads: 16
Started: Just now,Total memory: 120.00 GiB
Comm: tcp://127.0.0.1:45965,Total threads: 1
Dashboard: https://jupyterhub.hpc.ucar.edu/stable/user/jjeffree/more_testing/proxy/34599/status,Memory: 7.50 GiB
Nanny: tcp://127.0.0.1:36129,


2026-03-27 20:39:20,374 - distributed.scheduler - ERROR - Couldn't gather keys: {('open_dataset-ULONG-925909a1039aa1a995b6d0157f3a4360', 0, 0): 'forgotten'}
2026-03-27 20:39:50,263 - distributed.scheduler - ERROR - Couldn't gather keys: {('open_dataset-ULONG-a0abeec6d3a84e98b5f3488b8f24d15e', 0, 0): 'waiting'}
2026-03-27 20:40:16,219 - distributed.scheduler - ERROR - Couldn't gather keys: {('open_dataset-ULONG-e31e679072566cdac51ed191545baf8b', 0, 0): 'waiting'}
2026-03-27 20:40:19,599 - distributed.scheduler - ERROR - Couldn't gather keys: {('open_dataset-ULONG-aab1a54bc6343a250b8ca4c2b55618f8', 0, 0): 'waiting'}
2026-03-27 20:40:21,815 - distributed.scheduler - ERROR - Couldn't gather keys: {('open_dataset-ULONG-037c48aa58617672df6b339f1afdf1ed', 0, 0): 'waiting'}


In [3]:
def dcpp_tropical_nino34(ds,coords,tos_name='TEMP'):
    ''' VERY SPECIFIC TO CESM2 NOW'''
    ds = ds.isel(z_t=0)
    ds = ds.isel(time = slice(None, 98)) # I don't know why some of these runs seem to have been extended, but I don't want them
    ds = ds.drop_vars('time') # Gets in the way of concatenation
    return ds[tos_name].where((np.abs(ds[coords[1]]).compute()<=5)
                        & ((ds[coords[0]]%360).compute()<=240)
                        & ((ds[coords[0]]%360).compute()>=190)
                        ).mean(coords[2:4]).rename('nino34')

In [4]:
extension_dir = '/glade/campaign/cesm/development/espwg/CESM2-DP/timeseries/'
smyle_dir = '/glade/campaign/cesm/development/espwg/SMYLE/archive/'
out_dir = 'indices/'

In [5]:
Y = np.arange(1958,2019,dtype=int)
M = np.arange(11,31,dtype=int)
model = 'CESM2'
expt_dir = 'b.e21.BSMYLE.f09_g17.{Y}-11.{M}/'

giant_filepath_list = []
for y in Y:
    medium_filepath_list = []
    for m in M:

        two_files = (glob.glob(smyle_dir+expt_dir.format(Y=y,M=str(m).zfill(3),)+'ocn/proc/tseries/month_1/*.TEMP.*')
                     +glob.glob(extension_dir+expt_dir.format(Y=y,M=str(m).zfill(3),)+'ocn/proc/tseries/month_1/*.TEMP.*')
        )
        
        medium_filepath_list.append(two_files)
            
    giant_filepath_list.append(medium_filepath_list)

ds = xr.open_mfdataset(giant_filepath_list,
              combine = 'nested',
              concat_dim = ['Y','M','time'],
              parallel = True,
              chunks = {'nlat':-1,'nlon':-1,'time':-1}, # xarray doesn't seem to care about overprescribing chunks
              preprocess = partial(dcpp_tropical_nino34, coords=('TLONG','TLAT','nlon','nlat')),
              ).assign_coords({'M':M,'Y':Y}).load()
ds.to_netcdf(out_dir+model+'_DCPP_nino34.nc')

2026-03-27 20:39:20,375 - distributed.client - WARNING - Couldn't gather 1 keys, rescheduling (('open_dataset-ULONG-925909a1039aa1a995b6d0157f3a4360', 0, 0),)
2026-03-27 20:39:50,264 - distributed.client - WARNING - Couldn't gather 1 keys, rescheduling (('open_dataset-ULONG-a0abeec6d3a84e98b5f3488b8f24d15e', 0, 0),)
2026-03-27 20:40:16,220 - distributed.client - WARNING - Couldn't gather 1 keys, rescheduling (('open_dataset-ULONG-e31e679072566cdac51ed191545baf8b', 0, 0),)
2026-03-27 20:40:19,601 - distributed.client - WARNING - Couldn't gather 1 keys, rescheduling (('open_dataset-ULONG-aab1a54bc6343a250b8ca4c2b55618f8', 0, 0),)
2026-03-27 20:40:21,816 - distributed.client - WARNING - Couldn't gather 1 keys, rescheduling (('open_dataset-ULONG-037c48aa58617672df6b339f1afdf1ed', 0, 0),)
HDF5-DIAG: Error detected in HDF5 (1.14.3) thread 1:
  #000: H5F.c line 660 in H5Fcreate(): unable to synchronously create file
    major: File accessibility
    minor: Unable to create file
  #001: H5F.c 

PermissionError: [Errno 13] Permission denied: '/glade/u/home/jjeffree/ROM_ENSO/rom_decadal/u/home/jjeffree/ROM_ENSO/rom_decadal/indicesCESM2_DCPP_nino34.nc'

In [ ]:
ds.std('M').plot()

In [14]:
xr.open_mfdataset(glob.glob(extension_dir+expt_dir.format(Y=2018,M=str(16).zfill(3),)+'ocn/proc/tseries/month_1/*.TEMP.*'))

<xarray.Dataset> Size: 3GB
Dimensions:                 (moc_comp: 3, transport_comp: 5, transport_reg: 2,
                             z_t: 60, z_t_150m: 15, z_w: 60, z_w_top: 60,
                             z_w_bot: 60, lat_aux_grid: 395, moc_z: 61,
                             nlat: 384, nlon: 320, time: 98, d2: 2)
Coordinates:
  * z_t                     (z_t) float32 240B 500.0 1.5e+03 ... 5.375e+05
  * z_t_150m                (z_t_150m) float32 60B 500.0 1.5e+03 ... 1.45e+04
  * z_w                     (z_w) float32 240B 0.0 1e+03 ... 5e+05 5.25e+05
  * z_w_top                 (z_w_top) float32 240B 0.0 1e+03 ... 5e+05 5.25e+05
  * z_w_bot                 (z_w_bot) float32 240B 1e+03 2e+03 ... 5.5e+05
  * lat_aux_grid            (lat_aux_grid) float32 2kB -79.49 -78.95 ... 90.0
  * moc_z                   (moc_z) float32 244B 0.0 1e+03 ... 5.25e+05 5.5e+05
    ULONG                   (nlat, nlon) float64 983kB dask.array<chunksize=(384, 320), meta=np.ndarray>
    ULAT                    (nlat, nlon) float64 983kB dask.array<chunksize=(384, 320), meta=np.ndarray>
    TLONG                   (nlat, nlon) float64 983kB dask.array<chunksize=(384, 320), meta=np.ndarray>
    TLAT                    (nlat, nlon) float64 983kB dask.array<chunksize=(384, 320), meta=np.ndarray>
  * time                    (time) object 784B 2020-12-01 00:00:00 ... 2029-0...
Dimensions without coordinates: moc_comp, transport_comp, transport_reg, nlat,
                                nlon, d2
Data variables: (12/55)
    moc_components          (moc_comp) |S384 1kB dask.array<chunksize=(3,), meta=np.ndarray>
    transport_components    (transport_comp) |S384 2kB dask.array<chunksize=(5,), meta=np.ndarray>
    transport_regions       (transport_reg) |S384 768B dask.array<chunksize=(2,), meta=np.ndarray>
    dz                      (z_t) float32 240B dask.array<chunksize=(60,), meta=np.ndarray>
    dzw                     (z_w) float32 240B dask.array<chunksize=(60,), meta=np.ndarray>
    KMT                     (nlat, nlon) float64 983kB dask.array<chunksize=(384, 320), meta=np.ndarray>
    ...                      ...
    salinity_factor         float64 8B ...
    sflux_factor            float64 8B ...
    nsurface_t              float64 8B ...
    nsurface_u              float64 8B ...
    time_bound              (time, d2) object 2kB dask.array<chunksize=(1, 2), meta=np.ndarray>
    TEMP                    (time, z_t, nlat, nlon) float32 3GB dask.array<chunksize=(1, 30, 192, 160), meta=np.ndarray>
Attributes:
    title:             b.e21.BSMYLE.f09_g17.2018-11.016
    history:           none
    Conventions:       CF-1.0; http://www.cgd.ucar.edu/cms/eaton/netcdf/CF-cu...
    time_period_freq:  month_1
    model_doi_url:     https://doi.org/10.5065/D67H1H0V
    contents:          Diagnostic and Prognostic Variables
    source:            CCSM POP2, the CCSM Ocean Component
    revision:          $Id$
    calendar:          All years have exactly  365 days.
    start_time:        This dataset was created on 2021-06-08 at 14:31:55.2
    cell_methods:      cell_methods = time: mean ==> the variable values are ...

In [9]:
print(len(giant_filepath_list))
print(len(giant_filepath_list[0]))
print(len(giant_filepath_list[0][0]))

13
4
2


In [13]:
xr.open_mfdataset(/glade/campaign/cesm/development/espwg/CESM2-DP/timeseries/b.e21.BSMYLE.f09_g17.2018-11.016/ocn/proc/tseries/month_1/b.e21.BSMYLE.f09_g17.2018-11.016.pop.h.TEMP.202011-202812.nc')

SyntaxError: invalid decimal literal (3733276549.py, line 1)

In [16]:
for j,m in enumerate(M):
    for i,y in enumerate(Y):
        ds = xr.open_mfdataset(giant_filepath_list[i][j])
        if ds.time.shape[0]>122:
            assert False

AssertionError: 

In [20]:
y

np.int64(2018)

In [19]:
m

np.int64(16)

In [18]:
ds.time

<xarray.DataArray 'time' (time: 144)> Size: 1kB
array([cftime.DatetimeNoLeap(1958, 12, 1, 0, 0, 0, 0, has_year_zero=True),
       cftime.DatetimeNoLeap(1959, 1, 1, 0, 0, 0, 0, has_year_zero=True),
       cftime.DatetimeNoLeap(1959, 2, 1, 0, 0, 0, 0, has_year_zero=True),
       cftime.DatetimeNoLeap(1959, 3, 1, 0, 0, 0, 0, has_year_zero=True),
       cftime.DatetimeNoLeap(1959, 4, 1, 0, 0, 0, 0, has_year_zero=True),
       cftime.DatetimeNoLeap(1959, 5, 1, 0, 0, 0, 0, has_year_zero=True),
       cftime.DatetimeNoLeap(1959, 6, 1, 0, 0, 0, 0, has_year_zero=True),
       cftime.DatetimeNoLeap(1959, 7, 1, 0, 0, 0, 0, has_year_zero=True),
       cftime.DatetimeNoLeap(1959, 8, 1, 0, 0, 0, 0, has_year_zero=True),
       cftime.DatetimeNoLeap(1959, 9, 1, 0, 0, 0, 0, has_year_zero=True),
       cftime.DatetimeNoLeap(1959, 10, 1, 0, 0, 0, 0, has_year_zero=True),
       cftime.DatetimeNoLeap(1959, 11, 1, 0, 0, 0, 0, has_year_zero=True),
       cftime.DatetimeNoLeap(1959, 12, 1, 0, 0, 0, 0, has_year_zero=True),
       cftime.DatetimeNoLeap(1960, 1, 1, 0, 0, 0, 0, has_year_zero=True),
       cftime.DatetimeNoLeap(1960, 2, 1, 0, 0, 0, 0, has_year_zero=True),
       cftime.DatetimeNoLeap(1960, 3, 1, 0, 0, 0, 0, has_year_zero=True),
       cftime.DatetimeNoLeap(1960, 4, 1, 0, 0, 0, 0, has_year_zero=True),
       cftime.DatetimeNoLeap(1960, 5, 1, 0, 0, 0, 0, has_year_zero=True),
       cftime.DatetimeNoLeap(1960, 6, 1, 0, 0, 0, 0, has_year_zero=True),
       cftime.DatetimeNoLeap(1960, 7, 1, 0, 0, 0, 0, has_year_zero=True),
       cftime.DatetimeNoLeap(1960, 8, 1, 0, 0, 0, 0, has_year_zero=True),
       cftime.DatetimeNoLeap(1960, 9, 1, 0, 0, 0, 0, has_year_zero=True),
       cftime.DatetimeNoLeap(1960, 10, 1, 0, 0, 0, 0, has_year_zero=True),
       cftime.DatetimeNoLeap(1960, 11, 1, 0, 0, 0, 0, has_year_zero=True),
       cftime.DatetimeNoLeap(1960, 12, 1, 0, 0, 0, 0, has_year_zero=True),
       cftime.DatetimeNoLeap(1961, 1, 1, 0, 0, 0, 0, has_year_zero=True),
       cftime.DatetimeNoLeap(1961, 2, 1, 0, 0, 0, 0, has_year_zero=True),
       cftime.DatetimeNoLeap(1961, 3, 1, 0, 0, 0, 0, has_year_zero=True),
       cftime.DatetimeNoLeap(1961, 4, 1, 0, 0, 0, 0, has_year_zero=True),
       cftime.DatetimeNoLeap(1961, 5, 1, 0, 0, 0, 0, has_year_zero=True),
       cftime.DatetimeNoLeap(1961, 6, 1, 0, 0, 0, 0, has_year_zero=True),
       cftime.DatetimeNoLeap(1961, 7, 1, 0, 0, 0, 0, has_year_zero=True),
       cftime.DatetimeNoLeap(1961, 8, 1, 0, 0, 0, 0, has_year_zero=True),
       cftime.DatetimeNoLeap(1961, 9, 1, 0, 0, 0, 0, has_year_zero=True),
       cftime.DatetimeNoLeap(1961, 10, 1, 0, 0, 0, 0, has_year_zero=True),
       cftime.DatetimeNoLeap(1961, 11, 1, 0, 0, 0, 0, has_year_zero=True),
       cftime.DatetimeNoLeap(1961, 12, 1, 0, 0, 0, 0, has_year_zero=True),
       cftime.DatetimeNoLeap(1962, 1, 1, 0, 0, 0, 0, has_year_zero=True),
       cftime.DatetimeNoLeap(1962, 2, 1, 0, 0, 0, 0, has_year_zero=True),
       cftime.DatetimeNoLeap(1962, 3, 1, 0, 0, 0, 0, has_year_zero=True),
       cftime.DatetimeNoLeap(1962, 4, 1, 0, 0, 0, 0, has_year_zero=True),
       cftime.DatetimeNoLeap(1962, 5, 1, 0, 0, 0, 0, has_year_zero=True),
       cftime.DatetimeNoLeap(1962, 6, 1, 0, 0, 0, 0, has_year_zero=True),
       cftime.DatetimeNoLeap(1962, 7, 1, 0, 0, 0, 0, has_year_zero=True),
       cftime.DatetimeNoLeap(1962, 8, 1, 0, 0, 0, 0, has_year_zero=True),
       cftime.DatetimeNoLeap(1962, 9, 1, 0, 0, 0, 0, has_year_zero=True),
       cftime.DatetimeNoLeap(1962, 10, 1, 0, 0, 0, 0, has_year_zero=True),
       cftime.DatetimeNoLeap(1962, 11, 1, 0, 0, 0, 0, has_year_zero=True),
       cftime.DatetimeNoLeap(1962, 12, 1, 0, 0, 0, 0, has_year_zero=True),
       cftime.DatetimeNoLeap(1963, 1, 1, 0, 0, 0, 0, has_year_zero=True),
       cftime.DatetimeNoLeap(1963, 2, 1, 0, 0, 0, 0, has_year_zero=True),
       cftime.DatetimeNoLeap(1963, 3, 1, 0, 0, 0, 0, has_year_zero=True),
       cftime.DatetimeNoLeap(1963, 4, 1, 0, 0, 0, 0, has_year_zero=True),
       cftime.Dat

In [11]:
122-24

98

In [20]:
xr.open_dataset('/glade/campaign/cesm/development/espwg/SMYLE/archive/b.e21.BSMYLE.f09_g17.1958-11.011/ocn/proc/tseries/month_1/b.e21.BSMYLE.f09_g17.1958-11.011.pop.h.TEMP.195811-196010.nc')

<xarray.Dataset> Size: 728MB
Dimensions:                 (moc_comp: 3, transport_comp: 5, transport_reg: 2,
                             z_t: 60, z_t_150m: 15, z_w: 60, z_w_top: 60,
                             z_w_bot: 60, lat_aux_grid: 395, moc_z: 61,
                             nlat: 384, nlon: 320, time: 24, d2: 2)
Coordinates:
  * z_t                     (z_t) float32 240B 500.0 1.5e+03 ... 5.375e+05
  * z_t_150m                (z_t_150m) float32 60B 500.0 1.5e+03 ... 1.45e+04
  * z_w                     (z_w) float32 240B 0.0 1e+03 ... 5e+05 5.25e+05
  * z_w_top                 (z_w_top) float32 240B 0.0 1e+03 ... 5e+05 5.25e+05
  * z_w_bot                 (z_w_bot) float32 240B 1e+03 2e+03 ... 5.5e+05
  * lat_aux_grid            (lat_aux_grid) float32 2kB -79.49 -78.95 ... 90.0
  * moc_z                   (moc_z) float32 244B 0.0 1e+03 ... 5.25e+05 5.5e+05
    ULONG                   (nlat, nlon) float64 983kB ...
    ULAT                    (nlat, nlon) float64 983kB ...
    TLONG                   (nlat, nlon) float64 983kB ...
    TLAT                    (nlat, nlon) float64 983kB ...
  * time                    (time) object 192B 1958-12-01 00:00:00 ... 1960-1...
Dimensions without coordinates: moc_comp, transport_comp, transport_reg, nlat,
                                nlon, d2
Data variables: (12/55)
    moc_components          (moc_comp) |S384 1kB ...
    transport_components    (transport_comp) |S384 2kB ...
    transport_regions       (transport_reg) |S384 768B ...
    dz                      (z_t) float32 240B ...
    dzw                     (z_w) float32 240B ...
    KMT                     (nlat, nlon) float64 983kB ...
    ...                      ...
    salinity_factor         float64 8B ...
    sflux_factor            float64 8B ...
    nsurface_t              float64 8B ...
    nsurface_u              float64 8B ...
    time_bound              (time, d2) object 384B ...
    TEMP                    (time, z_t, nlat, nlon) float32 708MB ...
Attributes:
    title:             b.e21.BSMYLE.f09_g17.1958-11.011
    history:           none
    Conventions:       CF-1.0; http://www.cgd.ucar.edu/cms/eaton/netcdf/CF-cu...
    time_period_freq:  month_1
    model_doi_url:     https://doi.org/10.5065/D67H1H0V
    contents:          Diagnostic and Prognostic Variables
    source:            CCSM POP2, the CCSM Ocean Component
    revision:          $Id$
    calendar:          All years have exactly  365 days.
    start_time:        This dataset was created on 2022-03-23 at 02:51:35.3
    cell_methods:      cell_methods = time: mean ==> the variable values are ...

In [ ]:
/glade/campaign/cesm/development/espwg/SMYLE/archive/b.e21.BSMYLE.f09_g17.1958-11.011/ocn/proc/tseries/month_1

In [15]:
smyle_dir+filename.format(Y=y,M=str(m).zfill(3),date='*')

'/glade/campaign/cesm/development/espwg/SMYLE/archive/b.e21.BSMYLE.f09_g17.1958-11.011/*.nc'

In [12]:
str(m).zfill(3)

'011'